
# DELIVERABLE 3: CREATING A SMALL RPG

GitHub repository: https://github.com/Isfugl1/Deliverable_3.git

Video: 

#### Task 1: Update the Character class

For this task we updated the attack()-method so that it reduces the characters hp, this method has later been changed to use the take_damage()-method to reduce hp. The take_damage()-method also prints a message when used, so that we can keep a log of what happens during a battle (this can be viewed further down in the notebook). We also implemented a heal()-method that allows a character to regain hp. We arbitrarily set the base value as 5, since nothing was specified in the task. To make sure that the characters hp doesn't go below 0, we used pythons max()-function. It will pick the largest number between the reduced hp and 0. The same principle was used in the heal()-method to make sure a characters hp won't exceed the max. An is_alive()-property was also implemented. This property returns a bool, True if the character is alive and False if the characters hp is 0.

In [ ]:
# Object-oriented game simulator
# Turid Torheim, NMBU, 2026

# Imports the classes from the python files
from dice import Dice
from items import Weapon


class Character:
    """Represents a character in the RPG"""

    def __init__(
        self, name: str, hp_max: int = 100, damage: int = 10, healing: int = 5, weapon=None) -> None:
        """
        Initialise the Character
        Args:
        name (str): The name of the character.
        hp_max (int): Max hit points (default = 100)
        damage (int): Damage delt when character attacks (default = 10)
        healing (int): hp recovered (default = 5)
        weapon (Weapon or None): item a character can use to deal damage
        """
        self.name = name
        self._hp = hp_max  # Note: _hp indicated private attribute
        self.hp_max = hp_max
        self.damage = damage
        self.healing = healing
        self.weapon = weapon

    def take_damage(self, damage: int = 10) -> None:
        """Allows the characters to take damage"""
        if self._hp <= 0:
            print(f"The {self.name} is dead")
            return
        else:
            self._hp -= damage
            # Makes sure the hp doesn't go below zero
            self._hp = max(self._hp, 0)
        print(f"{self.name} took {damage} damage and has {self._hp} hp left")

    def heal(self, healing: int = 5) -> None:
        """Allows characters to heal"""
        self._hp += healing
        # Makes sure the hp doesn't go above the max
        self._hp = min(self._hp, self.hp_max)

    def attack(self, target) -> None:
        """Attack and deal damage to target character"""
        if self.weapon is not None:
            damage_from_weapon = self.weapon.roll_damage()
            target.take_damage(damage_from_weapon)
        else:
            target.take_damage(self.damage)

    def introduce(self) -> None:
        """Creates string that introduces character"""
        print(f"Hi, my name is {self.name}!")

    def __repr__(self) -> str:
        """Representation of character object"""
        return f"<Character name={self.name} hp={self.hp} damage={self.damage}>"

    def __str__(self) -> str:
        """String representation of character object"""
        return f"Character: {self.name} with {self.hp} HP and {self.damage} damage"

    @property
    def hp(self) -> int:
        """Public read-only property to inspect HP without allowing direct, unsafe
        edits."""
        return self._hp

    @property
    def is_alive(self) -> bool:
        """Checks if the character is alive"""
        return self._hp > 0

#### Task 2: Introducing randomness with a Dice class

For this task we created a dice class. This class has since been moved to a separate file (dice.py), but can be viewed in a text-format in this notebook. The random library  was used to generate a random number between 1 with the upper limit being the amount of sides the dice has. We used an if-statement to make sure the user can't give the dice less than 2 sides (coinflip). If this happens a ValueError is given with a message that explains the mistake. In addition to this to the spescial methods __repr__() and __call__() were added. The __repr__()-method creates a string that can be used by the developer (us) to see a representation of the dice object. __call__() allows an instance of the class to be called as a funtion.

import random


class Dice:

    def __init__(self, sides: int = 6) -> None:
        """Intialise the dice
        Args:
        sides (int): amount of sides the dice has
        """
        if sides < 2:
            raise ValueError("The die has to have at least two sides")
        self.sides = sides

    def roll(self) -> int:
        """Roll an n-sided dice"""
        self.eyes = random.randint(1, self.sides)
        return self.eyes

    def __repr__(self) -> str:
        """String representation of the dice"""
        return f"Dice(sides={self.sides})"

    def __call__(self) -> int:
        """Calls the roll()-method"""
        return self.roll()


#### Task 3: Create items

Like the code in the previous task, this code has also been moved to a separate file (items.py), but can be viewed bellow in a text-format. This class uses the Dice-class created in the previous task. This class allows the damage a character takes to be decided by a dice and not a set value. This allows the battle we will create later to have different outcomes based on these values.

from dice import Dice


class Weapon:

    """Represents a weapon a character can hold/equip"""

    def __init__(self, name: str, damage_dice: Dice) -> None:
        """initialise the weapon
        Args:
        name (str): what the weapon is called/what kind of weapon it is
        damage_dice (Dice): how much damage the weapon will deal
        """
        self.name = name
        self.damage_dice = damage_dice

    def roll_damage(self) -> int:
        """Uses the roll()-method from the Dice-class to decide damage"""
        return self.damage_dice.roll()

#### Task 4: Character specialization

In this task we used the Character-class to create two subclasses, a Warrior-class and a Mage-class. For the warrior we created an armor-parameter which reduces the incoming damage. Like in an earlier task we used pythons max()-function to make sure the damage doesn't go bellow 0 incase the armor-value/class is higher than the incoming damage.
More is done for the Mage-(sub)class. Here we introduce mana as a parameter that effects what kind of attack the character will use. In the initaliser we decided to use an if-statement to make sure a characters starting mana can't have a bigger value than the max (doing so gives a ValueError and an explaing message). We also decided to use an if-statemant to check if the mana was bigger or equal to 10. If the mage has 10 or more mana we double the basic damage value (since the task didn't ask for a spesific function or method for spell-attacks) and the mana gets reduced. If the mage has less than 10 mana it uses the 
attack()-method and increases the mana. To make sure the mana doesn't get a negative value og becomes larger than it's max, we use the max()- and min()-functions in the same way as earlier.

In [416]:
class Warrior(Character):
    def __init__(
        self, name, hp_max: int = 100, damage: int = 10, healing: int = 5, weapon=None, armor: int = 5
    ) -> None:
        """intialise the Warrrior-character
        Arg:
        super(): all arguments from the Character-class
        armor (int): gives the warrior armor that reduces damage
        """
        super().__init__(name, hp_max, damage, healing, weapon)
        self.armor = armor

    def take_damage(self, damage: int = 10) -> None:
        """Overrides the take_damage()-method in Character by reducing the damage by the armor-value/armorclass"""
        # Makes sure the damage doesn't go bellow zero
        real_damage = max(damage - self.armor, 0)
        super().take_damage(real_damage)


class Mage(Character):
    def __init__(
        self,
        name: str,
        hp_max: int = 100,
        damage: int = 10,
        healing: int = 5,
        weapon=None,
        mana_max: int = 30,
        mana: int = 30,
    ) -> None:
        """initialise the Mage-character
        Args:
        super(): all arguments from the Character class
        mana (int): the mages current amount of 'magical energy'
        mana_max (int): the max amount of mana the mage can have
        """
        super().__init__(name, hp_max, damage, healing, weapon)
        # The starting mana shouldn't be larger than the max
        if mana > mana_max:
            raise ValueError("The inital mana can not be higher than the maximum mana")
        self.mana = mana
        self.mana_max = mana_max

    def attack(self, target: Character) -> None:
        """Decides what kind of damage (melee or magical) and how much damage the mage will deal"""
        if self.mana >= 10:
            target.take_damage(self.damage * 2)
            # Makes sure the mana doesn't go bellow 0
            self.mana = max(self.mana - 10, 0)
        else:
            super().attack(target)
            # Makes sure the mana doesn't exceed the max
            self.mana = min(self.mana + 5, self.mana_max)


#### Task 5: Running battles

In this task we created a battle-class that uses the the methods we created earlier. The first method has the general structure for one round of a battle. In the second method we use play_round()-method in a while loop that check if the is_alive-property is True for both characters. To avoid a neverending loop we decided to use a round counter. In this code we draw line at 500 rounds, and declare the battle a stalemate. When one of the characters die, we use an if-statemnt to check which it is, and print an appropriate message. To make the battle somewhat more complicated we could use the heal()-method. It could for example be used to make the characters heal when their health is below a certain threshold or make the characters 'choose' between healing or dealing damage.

In [417]:
class Battle:

    def __init__(self, hero: Character, enemy: Character) -> None:
        """Initialise the Batte
        Args:
        hero (Character): heroic character
        enemy (Character): 'villan' character"""
        self.hero = hero
        self.enemy = enemy

    def play_round(self) -> None:
        """Creates the structure of one round in the battle"""
        self.hero.attack(self.enemy)
        if self.enemy.is_alive:
            self.enemy.attack(self.hero)

    def simulate_battle(self) -> None:
        """Simulates the battle and prints the outcome"""
        round_counter = 0
        while self.enemy.is_alive and self.hero.is_alive:
            self.play_round()
            round_counter += 1
            # If the given 'stats' for the characters results in a never-ending battle
            if round_counter >= 500:
                print("Stalemate!")
                return
        if self.hero.is_alive:
            print("The hero won!")
        else:
            print("The hero has been defeated!")

#### Do you wanna play a game?

Here we made an example battle using the methods and classes made in this deliverable. In this battle we have a "heroic" warrior and a "villainous" mage. 

In [418]:
d20 = Dice(sides=20)
sword = Weapon("sword", d20)

warrior = Warrior("Warrior", weapon=sword)
mage = Mage("Mage")

game = Battle(warrior, mage)
game.simulate_battle()

Mage took 16 damage and has 84 hp left
Warrior took 15 damage and has 85 hp left
Mage took 12 damage and has 72 hp left
Warrior took 15 damage and has 70 hp left
Mage took 18 damage and has 54 hp left
Warrior took 15 damage and has 55 hp left
Mage took 13 damage and has 41 hp left
Warrior took 5 damage and has 50 hp left
Mage took 18 damage and has 23 hp left
Warrior took 5 damage and has 45 hp left
Mage took 4 damage and has 19 hp left
Warrior took 15 damage and has 30 hp left
Mage took 11 damage and has 8 hp left
Warrior took 5 damage and has 25 hp left
Mage took 19 damage and has 0 hp left
The hero won!
